# Weather Join (Flights + Weather)

Join processed weather to cleaned BTS flights, then derive weather features.

## Inputs
- `data/processed/bts/flights_2024_clean.parquet`
- `data/processed/weather/weather_2024_hourly.parquet`

## Outputs
- `data/processed/integrated/flights_2024_weather.parquet`
- `data/reports/integrated/weather_join_quality_2024.csv`


## Step 0: Ensure project data (local or Hugging Face)

Downloads [ahiruuu/CIS-5450](https://huggingface.co/datasets/ahiruuu/CIS-5450) when `data/` is missing. Override with `HF_DATA_REPO_ID` if needed.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd().resolve()
for _p in [_here] + list(_here.parents):
    if (_p / "notebooks" / "project_data.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break

from project_data import ensure_project_data

ensure_project_data()

In [ ]:
import os
from pathlib import Path

import pandas as pd

from project_data import resolve_project_root

PROJECT_ROOT = resolve_project_root()
DATA_ROOT = Path(os.getenv("FLIGHT_DATA_DIR", PROJECT_ROOT / "data")).expanduser().resolve()
PROCESSED_BTS_DIR = DATA_ROOT / "processed" / "bts"
PROCESSED_WEATHER_DIR = DATA_ROOT / "processed" / "weather"
PROCESSED_INTEGRATED_DIR = DATA_ROOT / "processed" / "integrated"
REPORT_DIR = DATA_ROOT / "reports" / "integrated"
PROCESSED_INTEGRATED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

flights_path = PROCESSED_BTS_DIR / "flights_2024_clean.parquet"
weather_path = PROCESSED_WEATHER_DIR / "weather_2024_hourly.parquet"

if not flights_path.exists():
    raise FileNotFoundError(f"Missing input: {flights_path}")
if not weather_path.exists():
    raise FileNotFoundError(f"Missing input: {weather_path}")

flights = pd.read_parquet(flights_path)
weather = pd.read_parquet(weather_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Flights rows: {len(flights):,}")
print(f"Weather rows: {len(weather):,}")


In [ ]:
weather_cols = [
    "air_temp", "dew_point", "sea_level_pres", "wind_dir",
    "wind_speed", "sky_cover", "precip_1h", "precip_6h"
]

flights["FlightDate"] = pd.to_datetime(flights["FlightDate"], errors="coerce")
flights["CRSDepTime"] = pd.to_numeric(flights["CRSDepTime"], errors="coerce")
flights["dep_hour"] = (flights["CRSDepTime"] // 100).astype("Int64")
flights["dep_datetime"] = flights["FlightDate"] + pd.to_timedelta(flights["dep_hour"], unit="h")

weather_lookup = weather[["IATA", "obs_time"] + weather_cols].copy()

flights = flights.merge(
    weather_lookup.rename(columns={
        "IATA": "Origin",
        "obs_time": "dep_datetime",
        **{c: f"origin_{c}" for c in weather_cols}
    }),
    on=["Origin", "dep_datetime"],
    how="left"
)

flights = flights.merge(
    weather_lookup.rename(columns={
        "IATA": "Dest",
        "obs_time": "dep_datetime",
        **{c: f"dest_{c}" for c in weather_cols}
    }),
    on=["Dest", "dep_datetime"],
    how="left"
)

print(f"Rows after join: {len(flights):,}")
print(f"Origin weather missing rate: {flights['origin_air_temp'].isna().mean() * 100:.2f}%")
print(f"Dest weather missing rate:   {flights['dest_air_temp'].isna().mean() * 100:.2f}%")


In [ ]:
flights["origin_is_rain"] = (flights["origin_precip_1h"] > 0).astype(int)
flights["origin_high_wind"] = (flights["origin_wind_speed"] > 10).astype(int)
flights["origin_freezing"] = (flights["origin_air_temp"] < 0).astype(int)
flights["origin_low_vis"] = (flights["origin_sky_cover"] >= 7).astype(int)

flights["origin_weather_severity"] = (
    flights["origin_is_rain"] * 1.0 +
    flights["origin_high_wind"] * 1.5 +
    flights["origin_freezing"] * 1.2 +
    flights["origin_low_vis"] * 0.8
)

flights["worst_precip"] = flights[["origin_precip_1h", "dest_precip_1h"]].max(axis=1)
flights["worst_wind"] = flights[["origin_wind_speed", "dest_wind_speed"]].max(axis=1)

origin_missing_pct = float(round(flights["origin_air_temp"].isna().mean() * 100, 4))
dest_missing_pct = float(round(flights["dest_air_temp"].isna().mean() * 100, 4))

out_path = PROCESSED_INTEGRATED_DIR / "flights_2024_weather.parquet"
flights.to_parquet(out_path, index=False)

quality = pd.DataFrame([
    {
        "rows": int(len(flights)),
        "columns": int(len(flights.columns)),
        "origin_air_temp_missing_pct": origin_missing_pct,
        "dest_air_temp_missing_pct": dest_missing_pct,
    }
])
quality_path = REPORT_DIR / "weather_join_quality_2024.csv"
quality.to_csv(quality_path, index=False)

print(f"Saved integrated file: {out_path}")
print(f"Saved quality report: {quality_path}")
print(f"Final rows: {len(flights):,}")
print(f"Final columns: {len(flights.columns)}")
